# Endothelial cell identity literature

A 25-paper collection following the discovery, metadata organization and full-text retrieval steps in the reference's [Literature collection section](https://pmc.ncbi.nlm.nih.gov/articles/PMC12667862/#S8). SQLite and local files keep the prototype small.

The saved October 5, 2026 PubMed search returned 166 hits. Screening the first 60 relevance-ranked records produced the 25-paper sample. The inputs include all screening decisions and abstract-based annotations. This is not a systematic review.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import re
import sqlite3
import time
from collections import Counter
from pathlib import Path
from xml.etree import ElementTree as ET

import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
PMC_OAI = "https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/"
QUERY = ('endothelial[Title] AND (identity[Title/Abstract] OR specification[Title/Abstract] '
         'OR reprogramming[Title/Abstract] OR "cell fate"[Title/Abstract]) AND '
         '("transcription factor"[Title/Abstract] OR "transcription factors"[Title/Abstract] '
         'OR ETV2[Title/Abstract] OR ERG[Title/Abstract] OR FLI1[Title/Abstract] OR SOX17[Title/Abstract]) '
         'AND ("2015/01/01"[Date - Publication] : "2026/10/05"[Date - Publication]) '
         'NOT (Review[Publication Type] OR Editorial[Publication Type] OR Comment[Publication Type])')

def digest(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def utcnow():
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

def text(node):
    return " ".join("".join(node.itertext()).split()) if node is not None else ""

def write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False))

def get(url, params, allowed_statuses=()):

    for attempt in range(3):
        time.sleep(.4)
        try:
            r = requests.get(url, params=params, timeout=(15, 40),
                             headers={"User-Agent": "EndothelialLiteraturePrototype/0.1",
                                      "Accept-Encoding": "gzip, deflate"})
            if r.status_code == 429 or 500 <= r.status_code < 600:
                if attempt < 2:
                    retry = r.headers.get("Retry-After", "")

                    if retry.isdigit() and int(retry) > 30:
                        r.raise_for_status()
                    time.sleep(int(retry) if retry.isdigit() else 2 ** (attempt + 1))
                    continue
            if r.status_code not in allowed_statuses:
                r.raise_for_status()
            return r
        except (requests.Timeout, requests.ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2 ** (attempt + 1))
    raise RuntimeError("Request retries exhausted")

def discover(cache: Path, refresh=False):

    cache.mkdir(parents=True, exist_ok=True)
    search_path, records_path = cache / "literature.json", cache / "pubmed_records.xml"
    saved = json.loads(search_path.read_text())
    if refresh or not (search_path.exists() and records_path.exists()):
        r = get(EUTILS + "esearch.fcgi", {"db": "pubmed", "term": QUERY, "sort": "relevance",
                                         "retmax": 60, "retmode": "json"})
        response = r.json()
        ids = response["esearchresult"]["idlist"]
        if not ids:
            raise ValueError("No PubMed results")
        f = get(EUTILS + "efetch.fcgi", {"db": "pubmed", "id": ",".join(ids), "retmode": "xml"})
        parsed = parse_pubmed(f.content)
        if set(ids) != {p["pmid"] for p in parsed}:
            raise ValueError("EFetch did not return all requested IDs")
        records_path.write_bytes(f.content)
        saved["search"] = {"url": r.url, "query": QUERY, "retrieved_at_utc": utcnow(),
                                 "response": response, "efetch_url": f.url,
                                 "records_sha256": digest(f.content)}
        write_json(search_path, saved)
    provenance = saved["search"]
    if provenance["query"] != QUERY:
        raise ValueError("Cached search does not match the configured query")
    raw = records_path.read_bytes()
    if provenance.get("records_sha256") and provenance["records_sha256"] != digest(raw):
        raise ValueError("Cached PubMed XML hash mismatch")
    return parse_pubmed(raw), provenance

def parse_pubmed(raw):
    root = ET.fromstring(raw)
    if root.tag != "PubmedArticleSet":
        raise ValueError(f"Unexpected PubMed response: {root.tag}")
    papers = []
    for record in root.findall("PubmedArticle"):
        article = record.find("MedlineCitation/Article")

        ids = {x.attrib["IdType"]: x.text for x in record.findall("PubmedData/ArticleIdList/ArticleId")}
        pmid = record.findtext("MedlineCitation/PMID")
        abstract = " ".join(text(x) for x in article.findall("Abstract/AbstractText"))
        authors = []
        for author in article.findall("AuthorList/Author"):
            name = author.findtext("CollectiveName") or " ".join(filter(None, [author.findtext("ForeName"), author.findtext("LastName")]))
            authors.append({"name": name, "orcid": next((x.text for x in author.findall("Identifier") if x.get("Source") == "ORCID"), "")})
        journal = article.find("Journal")
        pubdate = journal.find("JournalIssue/PubDate")
        year = pubdate.findtext("Year") or (pubdate.findtext("MedlineDate", "")[:4])
        doi = (ids.get("doi") or "").strip().lower().removeprefix("https://doi.org/")
        papers.append({"pmid": pmid, "pmcid": ids.get("pmc", ""), "doi": doi,
                       "title": text(article.find("ArticleTitle")), "year": year,
                       "journal": journal.findtext("Title", ""), "issn": journal.findtext("ISSN", ""),
                       "authors": authors, "abstract": abstract,
                       "abstract_sha256": digest(abstract.encode()),
                       "publication_types": [text(x) for x in article.findall("PublicationTypeList/PublicationType")],
                       "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"})
    return papers

def deduplicate(papers):

    seen, keep, removed = set(), [], []
    for p in papers:
        keys = {(k, p[k]) for k in ["pmid", "doi"] if p[k]}
        keys.add(("title", re.sub(r"\W+", "", p["title"].casefold())))
        if keys & seen:
            removed.append(p["pmid"])
        else:
            keep.append(p)
        seen.update(keys)
    return keep, removed

def screen(papers, inputs: Path):
    saved = json.loads((inputs / "literature.json").read_text())
    decisions = saved["screening"]
    annotations = saved["annotations"]
    index = {d["pmid"]: d for d in decisions["records"]}
    ann = {a["pmid"]: a for a in annotations["records"]}
    if len(index) != len(decisions["records"]) or len(ann) != len(annotations["records"]):
        raise ValueError("Duplicate screening/annotation IDs")
    if set(index) != {p["pmid"] for p in papers}:
        raise ValueError("Candidate pool changed: update screening and annotations in inputs/literature.json")
    selected = []
    for paper in papers:
        decision = index[paper["pmid"]]
        if decision["include"]:
            a = ann[paper["pmid"]]
            if a["abstract_sha256"] != paper["abstract_sha256"]:
                raise ValueError(f"Changed abstract: re-review PMID {paper['pmid']}")
            if a["evidence_quote"] not in paper["abstract"]:
                raise ValueError(f"Ungrounded evidence quote in {paper['pmid']}")
            if len(a["evidence_quote"].split()) > 25:
                raise ValueError("Evidence excerpts must be short")
            selected.append({**paper, "annotation": a, "selection_reason": decision["reason"]})
    if set(ann) != {p["pmid"] for p in selected}:
        raise ValueError("Annotations and selected papers differ")
    if not 20 <= len(selected) <= 30:
        raise ValueError(f"Expected 20–30 papers, found {len(selected)}")
    return selected, decisions, annotations

In [2]:
def collect_fulltext(paper, cache: Path, online=False, receipt=None):

    if not paper["pmcid"]:
        return {"status": "no_pmcid", "license": "", "characters": 0}
    cache.mkdir(parents=True, exist_ok=True)
    pmcid = paper["pmcid"]
    status_path, xml_path, txt_path = (cache / f"{pmcid}{s}" for s in [".status.json", ".xml", ".txt"])
    if status_path.exists():
        status = json.loads(status_path.read_text())
        if status["status"] == "retrieved":
            if not xml_path.exists() or not txt_path.exists():
                if status.get("body_bundled") is False:
                    if not online:
                        return {**status, "body_available_locally": False}

                else:
                    raise ValueError(f"Incomplete full-text cache for {pmcid}")
            else:
                if digest(xml_path.read_bytes()) != status["xml_sha256"] or digest(txt_path.read_bytes()) != status["text_sha256"]:
                    raise ValueError(f"Corrupt full-text cache for {pmcid}")
                return {**status, "body_available_locally": True}
        else:
            return status
    if not online and receipt is not None:
        return {**receipt, "body_available_locally": False}
    if not online:
        return {"status": "not_attempted_offline", "license": "", "characters": 0}
    params = {"verb": "GetRecord", "identifier": "oai:pubmedcentral.nih.gov:" + pmcid.removeprefix("PMC"), "metadataPrefix": "pmc"}
    status = {"retrieved_at_utc": utcnow(), "endpoint": PMC_OAI, "params": params, "license": "", "characters": 0}
    try:

        r = get(PMC_OAI, params, allowed_statuses=(400,))
        root = ET.fromstring(r.content)
        error = root.find("{*}error")
        if error is not None:
            status.update(status="unavailable", reason=text(error), code=error.get("code"))
        else:
            article = root.find(".//{*}metadata/{*}article")
            if article is None:
                raise ValueError("No JATS article in response")

            for node in article.iter():
                node.tag = node.tag.split("}")[-1]
            ids = {x.get("pub-id-type"): x.text for x in article.findall("front/article-meta/article-id")}
            if ids.get("pmid") != paper["pmid"] or ids.get("pmcid", pmcid) != pmcid:
                raise ValueError("Full-text identifiers do not match the selected paper")
            body = article.find("body")
            if body is None or len(text(body)) < 500:
                status.update(status="no_body", reason="Metadata only or empty body")
            else:
                body_text = "\n\n".join(text(x) for x in body.iter() if x.tag in ["title", "p"])
                xml_path.write_bytes(r.content)
                txt_path.write_text(body_text)
                licenses = article.findall("front/article-meta/permissions/license")
                status.update(status="retrieved", source_url=r.url,
                              license=" | ".join(text(x) for x in licenses),
                              license_urls=[x.get("{http://www.w3.org/1999/xlink}href", "") for x in licenses],
                              characters=len(body_text), xml_sha256=digest(r.content), text_sha256=digest(txt_path.read_bytes()),
                              body_available_locally=True)
    except (requests.RequestException, ET.ParseError, ValueError) as e:
        status.update(status="error", reason=str(e))
    write_json(status_path, status)
    return status

In [3]:
def build_database(papers, path):

    with sqlite3.connect(path) as db:
        db.execute("PRAGMA foreign_keys=ON")
        for table in ["authorships", "papers", "journals"]:
            db.execute(f"DROP TABLE IF EXISTS {table}")
        db.execute("CREATE TABLE journals (journal_key TEXT PRIMARY KEY, title TEXT, issn TEXT)")
        db.execute("CREATE TABLE papers (pmid TEXT PRIMARY KEY, doi TEXT, pmcid TEXT, title TEXT, year TEXT, journal_key TEXT REFERENCES journals, abstract TEXT, category TEXT, factors TEXT, finding TEXT, evidence_quote TEXT, fulltext_status TEXT, license TEXT)")
        db.execute("CREATE TABLE authorships (pmid TEXT REFERENCES papers, position INTEGER, name TEXT, orcid TEXT, PRIMARY KEY (pmid, position))")
        for p in papers:
            j = p["issn"] or p["journal"]
            a = p["annotation"]
            db.execute("INSERT OR IGNORE INTO journals VALUES (?, ?, ?)", (j, p["journal"], p["issn"]))
            db.execute("INSERT INTO papers VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)", (p["pmid"], p["doi"], p["pmcid"], p["title"], p["year"], j, p["abstract"], a["category"], "; ".join(a["factors"]), a["finding"], a["evidence_quote"], p["fulltext"]["status"], p["fulltext"].get("license", "")))
            db.executemany("INSERT INTO authorships VALUES (?,?,?,?)", [(p["pmid"], i + 1, x["name"], x["orcid"]) for i, x in enumerate(p["authors"])])
        assert db.execute("PRAGMA foreign_key_check").fetchall() == []
        assert db.execute("SELECT count(*) FROM papers").fetchone()[0] == len(papers)

def search(papers, query, k=5):

    texts = [p["title"] + " " + p["abstract"] for p in papers]
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
    x = vectorizer.fit_transform(texts)
    scores = cosine_similarity(vectorizer.transform([query]), x).ravel()
    return [{"pmid": papers[i]["pmid"], "title": papers[i]["title"], "score": round(float(scores[i]), 4), "url": papers[i]["url"]}
            for i in scores.argsort()[::-1][:k] if scores[i] > 0]

def run(inputs=ROOT / "inputs", out=ROOT / "results/task2", fulltext_cache=ROOT / "data/pmc", online=False, refresh=False):
    inputs, out, fulltext_cache = map(Path, [inputs, out, fulltext_cache])
    out.mkdir(parents=True, exist_ok=True)
    candidates, provenance = discover(inputs, refresh=refresh)
    unique, removed = deduplicate(candidates)
    papers, decisions, annotations = screen(unique, inputs)
    receipts = json.loads((inputs / "literature.json").read_text()).get("fulltext", {})
    for p in papers:
        p["fulltext"] = collect_fulltext(p, fulltext_cache, online=online, receipt=receipts.get(p["pmcid"]))
        print(p["pmid"], p["fulltext"]["status"], flush=True)
    build_database(papers, out / "literature.sqlite")
    write_json(out / "papers.json", papers)
    columns = ["pmid", "doi", "pmcid", "year", "title", "journal", "category", "factors", "model", "methods", "finding", "limitation", "evidence_quote", "fulltext_status", "url"]
    with (out / "papers.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns); writer.writeheader()
        for p in papers:
            a = p["annotation"]
            row = {k: p[k] for k in ["pmid", "doi", "pmcid", "year", "title", "journal", "url"]}
            row.update({k: a[k] for k in ["category", "model", "finding", "limitation", "evidence_quote"]})
            row.update(factors="; ".join(a["factors"]), methods="; ".join(a["methods"]), fulltext_status=p["fulltext"]["status"])
            writer.writerow(row)
    examples = {q: search(papers, q) for q in ["ETV2 REST alternative lineage repression", "GATA4 MAF liver sinusoidal identity", "endothelial mesenchymal transition chromatin"]}
    write_json(out / "search_examples.json", examples)
    summary = {"topic": "Transcription-factor control of endothelial cell identity", "query": QUERY,
               "search_hits": int(provenance["response"]["esearchresult"]["count"]),
               "candidates_retrieved": len(candidates), "duplicates_removed": removed,
               "selected": len(papers), "excluded": len(unique) - len(papers),
               "categories": dict(Counter(p["annotation"]["category"] for p in papers)),
               "fulltext_status": dict(Counter(p["fulltext"]["status"] for p in papers)),
               "fulltext_bodies_available_locally": sum(p["fulltext"].get("body_available_locally", False) for p in papers),
               "all_annotation_hashes_and_quotes_validated": True,
               "human_review_status": "pending", "annotations_source": annotations["provenance"],
               "search_retrieved_at_utc": provenance["retrieved_at_utc"],
               "build_at_utc": utcnow(), "systematic_review": False}
    write_json(out / "summary.json", summary)
    print(json.dumps(summary, indent=2))
    return papers, summary

In [4]:
import pandas as pd
from IPython.display import display
from IPython.utils.capture import capture_output

ONLINE = False
REFRESH = False
with capture_output() as run_log:
    papers, summary = run(online=ONLINE, refresh=REFRESH)
display(pd.Series({k: summary[k] for k in ["search_hits", "candidates_retrieved", "selected", "excluded"]}).to_frame("papers"))
table = pd.read_csv("results/task2/papers.csv", dtype={"pmid": str})
display(table[["year", "pmid", "title", "category"]])
assert len(table) == 25 and table.pmid.nunique() == 25

,papers
search_hits,166
candidates_retrieved,60
selected,25
excluded,35


,year,pmid,title,category
0,2024,38503291,SOX17/ETV2 improves the direct reprogramming o...,Reprogramming
1,2020,33326796,Endothelial Reprogramming by Disturbed Flow Re...,Identity loss
2,2021,32916216,Endothelial GATA4 controls liver fibrosis and ...,Identity maintenance
3,2015,26537113,Venous Endothelial Marker COUP-TFII Regulates ...,Vascular specialization
4,2023,36512477,ETV2 Upregulation Marks the Specification of E...,Specification
5,2022,35550615,ETV2 functions as a pioneer factor to regulate...,Specification
6,2017,27778229,Production of endothelial progenitor cells fro...,Reprogramming
7,2015,26540589,Direct conversion of human amniotic cells into...,Reprogramming
8,2023,36512862,Sox17 mediates adult arterial endothelial cell...,Vascular specialization
9,2017,28055225,Overexpression of Prox1 Induces the Differenti...,Reprogramming


## Full text and search

The initial run retrieved 17 full texts. Four papers had no PMCID and four did not support export. The repository includes download records; article bodies stay out of the repository. Set `ONLINE = True` to fetch them again. Annotations and search use abstracts.

Set `REFRESH = True` to repeat discovery. Changed candidates need new screening and annotations. TF-IDF search ranks titles and abstracts by word overlap; it does not generate answers.

In [5]:
display(pd.Series(summary["fulltext_status"], name="papers").to_frame())
with sqlite3.connect("results/task2/literature.sqlite") as db:
    display(pd.read_sql_query("SELECT category, count(*) AS papers FROM papers GROUP BY category ORDER BY papers DESC", db))
results = search(papers, "ETV2 REST alternative lineage repression")
display(pd.DataFrame(results))
assert results[0]["pmid"] == "40495012"

,papers
retrieved,17
no_pmcid,4
unavailable,4


,category,papers
0,Specification,9
1,Reprogramming,6
2,Vascular specialization,4
3,Identity maintenance,4
4,Identity loss,2


,pmid,title,score,url
0,40495012,Pioneer factor ETV2 safeguards endothelial cel...,0.2171,https://pubmed.ncbi.nlm.nih.gov/40495012/
1,40832854,"The pioneer factor, ETV2, regulates networks t...",0.0634,https://pubmed.ncbi.nlm.nih.gov/40832854/
2,38570190,"ETV2 induces endothelial, but not hematopoieti...",0.0404,https://pubmed.ncbi.nlm.nih.gov/38570190/
3,25802403,Induction of hematopoietic and endothelial cel...,0.0345,https://pubmed.ncbi.nlm.nih.gov/25802403/
4,35550615,ETV2 functions as a pioneer factor to regulate...,0.0319,https://pubmed.ncbi.nlm.nih.gov/35550615/
